# Ensemble XAI on ImageNet-S

This notebook compares normalization and aggregation strategies for ensemble explainable AI (XAI). It uses a pretrained ResNet18 model, Captum attribution methods, and ImageNet-S segmentation masks.

**Author: Bipul Dutta.** This repository is a cleaned portfolio edition of the project.

> The evaluation section uses lightweight diagnostic proxies implemented in this notebook. These values are not official Quantus metric outputs and should not be interpreted as standardized benchmark scores.


## 1. Setup


In [ ]:
# Install the packages from requirements.txt before running this notebook.
import os
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from captum.attr import (
    Deconvolution,
    GradientShap,
    GuidedBackprop,
    InputXGradient,
    IntegratedGradients,
    NoiseTunnel,
    Saliency,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

BASE_DIR = Path(os.getenv("XAI_DATA_DIR", "data"))
INPUT_DIR = BASE_DIR / "input"
MASKS_DIR = INPUT_DIR / "ImageNetS50" / "train-semi-segmentation"
IMAGES_DIR = INPUT_DIR / "images"
RESULTS_DIR = Path("results")
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


## 2. Data preparation

Download the ImageNet-S50 masks from the official ImageNet-S release and place them under `data/input/ImageNetS50`. Download the matching ImageNet images through Kaggle's API after configuring your Kaggle credentials outside this repository. Credentials and datasets must never be committed.


In [ ]:
if not MASKS_DIR.exists():
    raise FileNotFoundError(
        f"ImageNet-S masks were not found at {MASKS_DIR}. "
        "See the README for dataset setup instructions."
    )

all_classes = sorted(path.name for path in MASKS_DIR.iterdir() if path.is_dir())
selected_classes = random.sample(all_classes, k=min(5, len(all_classes)))
print("Selected classes:", selected_classes)


In [ ]:
class ImageNetSDataset(Dataset):
    def __init__(self, classes):
        self.samples = []
        self.image_transform = T.Compose([T.Resize((224, 224)), T.ToTensor()])
        self.mask_transform = T.Compose([
            T.Resize((224, 224), interpolation=T.InterpolationMode.NEAREST),
            T.ToTensor(),
        ])

        for class_id in classes:
            mask_dir = MASKS_DIR / class_id
            image_dir = IMAGES_DIR / class_id
            for mask_path in sorted(mask_dir.glob("*.png")):
                image_path = image_dir / f"{mask_path.stem}.JPEG"
                if image_path.exists():
                    self.samples.append((image_path, mask_path, class_id))
                    break

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path, mask_path, class_id = self.samples[index]
        image = Image.open(image_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")
        return self.image_transform(image), self.mask_transform(mask), class_id


dataset = ImageNetSDataset(selected_classes)
if not dataset:
    raise RuntimeError("No matching image-mask pairs were found. See README.md.")
loader = DataLoader(dataset, batch_size=1, shuffle=False)
print("Dataset size:", len(dataset))


## 3. Model and attribution methods


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1).to(device)
model.eval()


def generate_explanations(model, image, target):
    image = image.unsqueeze(0).to(device)
    zero_baseline = torch.zeros_like(image)
    return {
        "Integrated Gradients": IntegratedGradients(model).attribute(image, target=target),
        "Saliency": Saliency(model).attribute(image, target=target),
        "Guided Backprop": GuidedBackprop(model).attribute(image, target=target),
        "Input x Gradient": InputXGradient(model).attribute(image, target=target),
        "Deconvolution": Deconvolution(model).attribute(image, target=target),
        "GradientSHAP": GradientShap(model).attribute(image, target=target, baselines=zero_baseline),
        "Noise Tunnel": NoiseTunnel(IntegratedGradients(model)).attribute(
            image, nt_samples=8, target=target
        ),
    }


## 4. Normalization and aggregation


In [ ]:
def to_map(value):
    value = value.detach().cpu()
    if value.ndim == 4:
        value = value.squeeze(0)
    if value.ndim == 3:
        value = value.mean(dim=0)
    return value


def standardize(value):
    return (value - value.mean()) / (value.std() + 1e-6)


def robust_standardize(value):
    median = value.median()
    iqr = torch.quantile(value, 0.75) - torch.quantile(value, 0.25)
    return (value - median) / (iqr + 1e-6)


def second_moment_scale(value):
    return value / (torch.sqrt(torch.mean(value ** 2)) + 1e-6)


NORMALIZERS = {
    "Standard": standardize,
    "Robust": robust_standardize,
    "Second moment": second_moment_scale,
    "Absolute": torch.abs,
}


def aggregate(maps, method):
    stack = torch.stack(maps)
    if method == "Mean":
        return stack.mean(dim=0)
    if method == "Median":
        return stack.median(dim=0).values
    if method == "Geometric mean":
        return torch.exp(torch.log(torch.abs(stack) + 1e-6).mean(dim=0))
    raise ValueError(f"Unknown aggregation method: {method}")


AGGREGATORS = ["Mean", "Median", "Geometric mean"]


The original experiment included a "weighted mean" with equal weights. Equal weighting is mathematically identical to the ordinary mean, so the duplicate configuration is omitted here.


In [ ]:
records = []
visual_examples = []

for image, mask, class_id in tqdm(loader):
    image = image.to(device)
    with torch.no_grad():
        target = model(image).argmax(dim=1).item()

    raw = generate_explanations(model, image[0], target)
    base_maps = [to_map(value) for value in raw.values()]

    for norm_name, normalizer in NORMALIZERS.items():
        normalized = [normalizer(value) for value in base_maps]
        for agg_name in AGGREGATORS:
            combined = aggregate(normalized, agg_name)
            records.append({
                "class_id": class_id[0],
                "normalization": norm_name,
                "aggregation": agg_name,
                "map": combined,
                "mask": mask[0, 0],
            })

    visual_examples.append((image.cpu(), class_id[0], raw))

print(f"Created {len(records)} ensemble maps.")


## 5. Visualization


In [ ]:
def display_map(value):
    array = np.abs(to_map(value).numpy())
    array = cv2.GaussianBlur(array, (0, 0), sigmaX=6)
    array -= array.min()
    return array / (array.max() + 1e-8)


for record in records[:3]:
    plt.figure(figsize=(5, 4))
    plt.imshow(display_map(record["map"]), cmap="jet")
    plt.title(f'{record["normalization"]} + {record["aggregation"]}')
    plt.axis("off")
    plt.tight_layout()
    plt.show()


## 6. Lightweight diagnostic proxies

The following functions are transparent, project-specific diagnostics. They are useful for comparing runs inside this notebook, but they are **not** official Quantus implementations of faithfulness, randomization, robustness, complexity, or localization metrics.


In [ ]:
def attribution_magnitude_proxy(attr_map):
    return float(np.abs(to_map(attr_map).numpy()).mean())


def localization_top_k_proxy(attr_map, mask, k_percent=5):
    attribution = np.abs(to_map(attr_map).numpy())
    mask_array = np.asarray(mask > 0.5, dtype=bool)
    k = max(1, int((k_percent / 100) * attribution.size))
    top_indices = np.argpartition(attribution.ravel(), -k)[-k:]
    return float(mask_array.ravel()[top_indices].mean())


rows = []
for record in records:
    rows.append({
        "class_id": record["class_id"],
        "normalization": record["normalization"],
        "aggregation": record["aggregation"],
        "attribution_magnitude_proxy": attribution_magnitude_proxy(record["map"]),
        "localization_top_5_percent_proxy": localization_top_k_proxy(
            record["map"], record["mask"]
        ),
    })

results = pd.DataFrame(rows)
summary = (
    results.groupby(["normalization", "aggregation"], as_index=False)
    .mean(numeric_only=True)
)
summary.to_csv(RESULTS_DIR / "diagnostic_summary.csv", index=False)
summary


## 7. Next steps

- Replace the diagnostic proxies with documented Quantus metric classes and explicit model/data wrappers.
- Evaluate more images and report uncertainty across repeated seeded samples.
- Compare additional architectures and perturbation-based explainers.
